#  Predicting Airline Flight Delays Using Machine Learning Regression

---

##  Project Overview

| Field | Details |
|---|---|
| **Project Title** | Predicting Airline Flight Delays Using Machine Learning Regression |
| **Domain** | Aviation / Transportation |
| **Task Type** | Supervised Learning — Regression |
| **Target Variable** | `ArrDelay` (Arrival Delay in minutes) |

---

##  Problem Statement

Flight delays are a persistent and costly problem in the airline industry, directly impacting passenger experience and airline operations. Several factors drive these delays — weather, air traffic congestion, carrier-side issues, and late arriving aircraft. The goal of this project is to analyze historical flight data and build a regression-based machine learning model that can predict how long a flight will be delayed (in minutes). Understanding the key factors behind delays can help airlines improve scheduling, resource allocation, and overall operational performance.

---

##  Business Objective

Build a reliable predictive model that estimates flight arrival delays using historical airline data. The model should help airlines and airport management teams make data-driven decisions — improving operational planning, reducing unnecessary delays, and enhancing the overall passenger experience through better communication and delay management.

---

## 🌻 TASK 1 — Data Acquisition & Initial Exploration

---

### Step 1 — Import Libraries

We begin by importing all the essential libraries needed throughout this project. These cover data manipulation, visualization, modeling, and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

---

### Step 2 — Load the Dataset

We load the raw flight delay dataset from a CSV file and take a quick look at the first few rows to understand its structure.

In [ ]:
df = pd.read_csv("Flight_delay.csv")
df.head(2)

In [ ]:
# Check how many records are in the full dataset
df.shape  

In [ ]:
# We work with the first 70,000 rows to keep things manageable
data = df.head(70000)

In [ ]:
# Confirm the shape of our working dataset
data.shape

In [ ]:
# List all column names
data.columns

In [ ]:
# Rearrange columns so the target variable 'ArrDelay' is at the end
cols = [col for col in data.columns if col != 'ArrDelay'] + ['ArrDelay']
data = data[cols]

In [ ]:
# Final check after reordering — target column should now be last
data.head(2)  

---

### 📊 Dataset Description

| Property | Detail |
|---|---|
| **Rows** | 70,000 |
| **Columns** | 29 |
| **Target Variable** | `ArrDelay` |
| **Prediction Task** | Regression — predicting a continuous value (delay in minutes) |

> We are dealing with a regression problem, not classification, because the target `ArrDelay` is a continuous numerical value.

In [ ]:
# Get a concise summary: column names, data types, and non-null counts
data.info()

---

## 🌻 TASK 2 — Data Preprocessing

Data preprocessing is the foundation of any machine learning project. Raw data is almost never clean — it contains missing values, duplicates, inconsistencies, and irrelevant features. We handle all of these step by step.

---

### Step 3 — Data Cleaning

#### 3.1 — Checking and Handling Missing Values

We first check which columns have missing values and how many.

In [ ]:
# Count missing values in each column
data.isnull().sum() 

In [ ]:
# Inspect the 'Org_Airport' column — checking how many unique values it has
df['Org_Airport'].value_counts()

In [ ]:
# Inspect the 'Dest_Airport' column similarly
df['Dest_Airport'].value_counts()

In [ ]:
# Both airport columns have too many unique categorical values with no clean
# encoding path — dropping them to avoid noise in the model
data.drop(['Org_Airport', 'Dest_Airport'], axis=1, inplace=True)

In [ ]:
# Confirm the shape after dropping the two airport columns
data.shape

In [ ]:
# Re-check missing values after cleanup
data.isnull().sum()

---

#### 3.2 — Handling Duplicate Rows

Duplicate records can mislead the model and inflate certain patterns in the data.

In [ ]:
# Count the number of exact duplicate rows
duplicates = data.duplicated().sum()
duplicates

In [ ]:
# Remove all duplicate rows in place
data.drop_duplicates(inplace=True)

In [ ]:
# Confirm the shape after removing duplicates
data.shape

---

#### 3.3 — Handling Data Inconsistencies and Errors

We now check for inconsistent labels, wrong data types, unnecessary columns, and any other issues that could hurt model quality.

In [ ]:
# Check data types of all columns
data.dtypes

In [ ]:
# Identify all object (text/categorical) type columns
data.select_dtypes(include='object').columns

In [ ]:
# Identify all numeric type columns
data.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
# Check the number of unique values per column — helps spot inconsistent labels
print("\nUnique Values in Each Column:")
for col in data.columns:
    print(f"{col}: {data[col].nunique()} unique")

In [ ]:
# 'TailNum' is an aircraft registration number — highly unique, not useful for modeling
data.value_counts("TailNum")  # check for every columns

In [ ]:
# Drop 'TailNum' — too many unique values, no predictive value
data.drop('TailNum', axis=1, inplace=True)

In [ ]:
# Convert 'Date' to datetime and extract Year and Month as separate features
# This allows the model to capture seasonal patterns
data['Date'] = pd.to_datetime(data['Date'], dayfirst=True)

data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month

In [ ]:
# Drop the original 'Date' column since Year and Month now carry the needed info
data.drop('Date', axis=1, inplace=True)

**Note on `unique()` vs `value_counts()` in EDA:**

- `df.unique()` — quickly shows what categories exist; great for spotting typos or inconsistent labels.
- `df.value_counts()` — shows how often each category appears; useful for measuring class imbalance or rare values.

Best practice: inspect with `.unique()` first, then use `.value_counts()` to measure and decide how to fix any issues.

In [ ]:
# Print unique values for all categorical (object) columns to spot inconsistencies
print("\nPotential Inconsistent Categorical Values:")
cat_cols = data.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"\n{col}:", data[col].unique())

In [ ]:
# Check how many unique origin airports exist in the data
data['Origin'].nunique()

In [ ]:
# 'CancellationCode' has very few unique values — mostly one dominant value
# A feature where one value dominates is not useful for a model
data.value_counts("CancellationCode")

In [ ]:
# Confirm the unique values in 'Cancelled' column
data["Cancelled"].unique()

In [ ]:
# Drop 'CancellationCode' — it provides no useful discriminating power
data.drop('CancellationCode', axis=1, inplace=True)